# Person 6 — Deployment, inference, and final reporting

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Load only the trusted local model, reproduce the exact Person 3 features for a new image, test deployment compatibility, and assemble the final six-person report. Maintain the Flask website and communicate limitations.

**Input:** All previous artifacts  
**Output:** `06_final_report.json`; optional prediction for `NEW_IMAGE`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)
import joblib
from PIL import Image, ImageOps
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog, local_binary_pattern

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
def handcrafted(image):
    rgb=np.asarray(image.resize((128,128),Image.Resampling.BILINEAR),dtype=np.float32)/255; hsv=rgb2hsv(rgb); parts=[]
    for array in (rgb,hsv):
        for channel in range(3):
            h,_=np.histogram(array[:,:,channel],bins=16,range=(0,1)); parts.append(h.astype(np.float32)/h.sum())
        parts.extend([array.mean(axis=(0,1)),array.std(axis=(0,1))])
    gray=(rgb2gray(rgb)*255).astype(np.uint8); lbp=local_binary_pattern(gray,8,1,method='uniform'); h,_=np.histogram(lbp,bins=np.arange(11)); parts.append(h.astype(np.float32)/h.sum())
    small=np.asarray(image.resize((64,64)).convert('L'),dtype=np.float32)/255; parts.append(hog(small,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2)))
    return np.concatenate([np.ravel(x) for x in parts]).astype(np.float32)
bundle=joblib.load(ARTIFACTS/'05_model.joblib') # Load only the artifact created by Person 5.
if bundle['feature_version']!='rgb-hsv-lbp-hog-v1': raise ValueError('Feature version mismatch')
collection=json.loads((ARTIFACTS/'01_collection_report.json').read_text()); preprocessing=json.loads((ARTIFACTS/'02_preprocessing_report.json').read_text()); features=json.loads((ARTIFACTS/'03_feature_report.json').read_text()); selection=json.loads((ARTIFACTS/'04_selection.json').read_text()); evaluation=json.loads((ARTIFACTS/'05_evaluation.json').read_text())
final={'pipeline_parts':6,'dataset':collection,'preprocessing':preprocessing,'features':features,'selection':selection,'evaluation':evaluation,'deployment':{'interface':'Flask website at http://127.0.0.1:8000','prediction_flow':'upload -> safe RGB decode -> 1,882 features -> saved Random forest -> label','retraining':'Upload does not retrain. Run Persons 1–5 after adding data, then restart the website.'},'limitations':['Partial IEEE download; missing sources are unrepresented.','Whole-image features can learn background, camera or lighting.','No independent new-farm/new-plant validation.','Prediction is not disease diagnosis or food-safety advice.','No calibrated confidence or unrelated-object rejection.']}
(ARTIFACTS/'06_final_report.json').write_text(json.dumps(final,indent=2),encoding='utf-8')
print('Final report saved. Test accuracy:',f"{evaluation['accuracy']:.2%}")
NEW_IMAGE='' # Example: r'C:\Users\herat\Pictures\basil.jpg'
if NEW_IMAGE:
    with Image.open(NEW_IMAGE) as opened: image=ImageOps.exif_transpose(opened).convert('RGB')
    vector=handcrafted(image)[None,:]; print('Prediction:',bundle['estimator'].predict(vector)[0]); display(image.resize((300,max(16,int(300*image.height/image.width)))))
else: print('Set NEW_IMAGE to test another photo.')

Final report saved. Test accuracy: 95.56%
Set NEW_IMAGE to test another photo.


## Final communication checklist
Report the source, actual unique-image count, class balance, leakage controls, feature design, complete candidate comparison, selection rule, final test metrics, confusion matrix, limitations, and deployment flow. Never present this prototype as a medical, agricultural, or food-safety diagnosis.